# Choveu mais ou menos que o normal na minha cidade? — versão Google Colab

**O problema:** num ano de El Niño forte, a chuva sai do padrão. Mas "choveu pouco"
só quer dizer alguma coisa quando você compara com o histórico da sua região.

**O que você vai fazer:** baixar os dados de uma estação do INMET, somar a chuva de
cada mês de 2026 e comparar com a média dos anos anteriores — tudo de graça, sem
instalar nada no seu computador.

**Fonte do dado:** INMET, Dados Históricos (estações automáticas) — https://portal.inmet.gov.br/dadoshistoricos

⏱️ Tempo: ~10–15 minutos (a primeira rodada baixa os arquivos) · Nível: iniciante

> Como usar: clique em cada célula e aperte **Shift + Enter**, uma de cada vez, de cima para baixo.

## 1. Conectar ao Google Drive
Assim os arquivos baixados e os resultados ficam salvos no seu Drive e não somem quando
você fechar o Colab. Na segunda vez que rodar, ele reaproveita o que já baixou. O Colab
vai pedir autorização: aceite.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/cases-agro/chuva-vs-media"
CACHE = f"{PASTA}/inmet_zips"
os.makedirs(CACHE, exist_ok=True)
print("Resultados serão salvos em:", PASTA)

## 2. ⚙️ Configure aqui
Troque os valores abaixo para a **sua** estação e período. **É a única célula que você
precisa mexer.**

- `ESTACAO`: o código da estação automática mais perto de você (ex.: `A904` é Sorriso/MT).
  Veja o mapa e a lista em https://mapas.inmet.gov.br → clique na estação para ver o código.
- `ANO_INICIO`: de que ano começa a sua "média". Quanto mais anos, mais confiável — e mais
  demorado para baixar na primeira vez. Aqui usamos 2021 para o exemplo rodar rápido.

In [ ]:
ESTACAO = "A904"            # código da estação (troque pelo da sua cidade)
NOME = "Sorriso/MT (A904)"  # só para aparecer no título do gráfico
ANO_INICIO = 2021
ANO_ALVO = 2026             # o ano que você quer comparar com a média

## 3. Ferramentas
O Colab já vem com quase tudo. Aqui só importamos as bibliotecas.

In [ ]:
import io
import zipfile

import pandas as pd
import requests
import matplotlib.pyplot as plt

MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

## 4. Baixar os dados
O INMET publica **um arquivo .zip por ano**, com todas as estações do Brasil. Baixamos
os anos que você escolheu (os que já estiverem no seu Drive não são baixados de novo).

In [ ]:
URL = "https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"

def baixar_ano(ano):
    destino = f"{CACHE}/{ano}.zip"
    if os.path.exists(destino):
        return destino
    print(f"Baixando {ano}... (~100 MB)")
    with requests.get(URL.format(ano=ano), stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloco in r.iter_content(1 << 20):
                f.write(bloco)
    return destino

anos = list(range(ANO_INICIO, ANO_ALVO + 1))
for ano in anos:
    baixar_ano(ano)
print("Pronto. Anos disponíveis:", anos)

## 5. Ler e limpar uma estação
Cada arquivo do INMET tem **8 linhas de cabeçalho** (região, estação, latitude...) antes
da tabela. Os dados são **de hora em hora**, o separador é `;` e o decimal é vírgula.
Onde não houve medição, aparece `-9999` — isso vira "vazio" para não bagunçar a conta.

⚠️ O INMET **mudou o cabeçalho ao longo dos anos**: em arquivos antigos a coluna de data se
chama `DATA (YYYY-MM-DD)` e vem com traço (`2016-01-01`); nos novos é `Data` com barra
(`2026/01/01`). Por isso aqui pegamos a **primeira coluna** como data e deixamos o pandas
descobrir o formato — assim nenhum ano é descartado por engano.

In [ ]:
def chuva_mensal(ano):
    """Soma a chuva por mês (1 a 12) da estação escolhida, no ano dado."""
    with zipfile.ZipFile(f"{CACHE}/{ano}.zip") as z:
        nomes = [n for n in z.namelist() if f"_{ESTACAO}_" in n]
        if not nomes:
            print(f"  aviso: estação {ESTACAO} não existe no arquivo de {ano}")
            return pd.Series(dtype=float)
        with z.open(sorted(nomes)[-1]) as f:   # pega o arquivo de maior cobertura do ano
            bruto = io.BytesIO(f.read())
    raw = pd.read_csv(bruto, sep=";", decimal=",", encoding="latin-1", skiprows=8)
    col_data = raw.columns[0]                                      # a 1ª coluna é sempre a data
    col_prec = next(c for c in raw.columns if "PRECIPITA" in c.upper())
    df = raw[[col_data, col_prec]].copy()
    df.columns = ["data", "precip_mm"]
    df["precip_mm"] = pd.to_numeric(df["precip_mm"], errors="coerce")
    df.loc[df["precip_mm"] < 0, "precip_mm"] = pd.NA               # -9999 = sem medição
    df["data"] = pd.to_datetime(df["data"], errors="coerce")       # entende traço e barra
    return df.groupby(df["data"].dt.month)["precip_mm"].sum()

# Testa com o ano-alvo:
chuva_mensal(ANO_ALVO)

## 6. Montar a comparação: 2026 × média histórica
Somamos a chuva de cada mês, ano a ano. A "média histórica" é a média de cada mês nos
anos anteriores a 2026. Só comparamos os meses que já têm dado em 2026 (os meses já
fechados).

In [ ]:
alvo = chuva_mensal(ANO_ALVO)
ref = pd.concat([chuva_mensal(a) for a in anos if a < ANO_ALVO], axis=1).mean(axis=1)

tabela = pd.DataFrame({
    "mes": [MESES[m - 1] for m in alvo.index],
    "media_historica_mm": [round(ref.get(m, 0), 1) for m in alvo.index],
    f"{ANO_ALVO}_mm": [round(alvo[m], 1) for m in alvo.index],
})
tabela["anomalia_mm"] = (tabela[f"{ANO_ALVO}_mm"] - tabela["media_historica_mm"]).round(1)
tabela

## 7. Visualizar
Um gráfico com uma mensagem só: onde a barra verde passa da cinza, choveu **mais** que o
normal; onde fica abaixo, choveu **menos**.

In [ ]:
import numpy as np

x = np.arange(len(tabela))
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - 0.21, tabela["media_historica_mm"], width=0.42, label=f"Média {ANO_INICIO}–{ANO_ALVO-1}")
ax.bar(x + 0.21, tabela[f"{ANO_ALVO}_mm"], width=0.42, label=str(ANO_ALVO))
ax.set_xticks(x); ax.set_xticklabels(tabela["mes"])
ax.set_ylabel("chuva no mês (mm)")
ax.set_title(f"Choveu mais ou menos que o normal? {NOME}")
ax.legend()
plt.show()

## 8. Salvar no Drive

In [ ]:
tabela.to_csv(f"{PASTA}/anomalia_{ESTACAO}.csv", index=False)
fig.savefig(f"{PASTA}/anomalia_{ESTACAO}.png", dpi=150, bbox_inches="tight")
print("Arquivos salvos em", PASTA)

## 9. Como você usa isso amanhã
- Troque `ESTACAO` pela estação da sua cidade e veja o padrão da sua região.
- Olhe os meses de plantio: um mês muito abaixo da média acende o alerta de veranico.
- Compare vizinhos: rode para 2 ou 3 estações e veja se a anomalia é local ou regional.

- **Cuidado 1:** esta "média" é a dos últimos anos das estações automáticas, **não** é a
  Normal Climatológica de 30 anos do INMET. Quanto mais anos você usar, melhor.
- **Cuidado 2:** 2026 só tem meses fechados até o último disponível no arquivo. Meses
  incompletos não entram na comparação.
- **Cuidado 3:** estação com muita falha (`-9999`) subestima a chuva do mês. Se um mês
  parecer estranho, confira quantas horas faltaram.

Gostou? Me conta no Instagram **@adrielpereira.eng** qual estação você testou.